In [1]:
# Scenario: University Library Assistant
# A large university library has thousands of digitized textbooks, research papers, and course notes. Students often struggle to find specific explanations or summaries when preparing for exams. Instead of manually searching through PDFs, the library deploys a RAG chatbot that acts like a study companion.
# How It Works
# - Input Source: Students upload or access a textbook PDF (e.g., Introduction_to_Data_Science.pdf).
# - Chunking: The chatbot splits the textbook into smaller sections so that each concept is searchable.
# - Embeddings + Vector DB: Each section is embedded and stored in Chroma, making the textbook searchable by meaning.
# - Retriever: When a student asks, “Explain the difference between supervised and unsupervised learning,” the retriever pulls the most relevant sections.
# - LLM Response: The Hugging Face model generates a clear, concise answer tailored to the student’s query.
# - Interactive Chat: Students can keep asking follow-up questions, turning the textbook into a conversational tutor.


In [2]:
# ==========================================================
# UNIVERSITY LIBRARY RAG ASSISTANT
# PDF + CHROMA + EMBEDDINGS + HUGGINGFACE LLM
# ==========================================================

# ----------------------------------------------------------
# STEP 0 — Install Required Libraries
# ----------------------------------------------------------
# pip install chromadb sentence-transformers pypdf transformers
!pip install gradio chromadb sentence-transformers pypdf transformers python-dotenv mistralai

Defaulting to user installation because normal site-packages is not writeable


In [3]:
# ----------------------------------------------------------
# STEP 1 — Import Libraries
# ----------------------------------------------------------

import os
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
import os
from mistralai.client import Mistral
from dotenv import load_dotenv
load_dotenv()

C:\Users\sk165\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [4]:
# ----------------------------------------------------------
# STEP 2 — Load LLM (HuggingFace Flan-T5)
# ----------------------------------------------------------

print("Loading HuggingFace model...")

llm = Mistral(api_key=os.environ['MISTRAL_API_KEY'])
print("LLM loaded successfully")

Loading HuggingFace model...
LLM loaded successfully


In [5]:
# ----------------------------------------------------------
# STEP 3 — Load Embedding Model
# ----------------------------------------------------------

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3060.10it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded


In [6]:
# ----------------------------------------------------------
# STEP 4 — Initialize Vector Database
# ----------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("library_docs")
except:
    pass

collection = client.create_collection("library_docs")

In [7]:
# ----------------------------------------------------------
# STEP 5 — Load PDF
# ----------------------------------------------------------

print("Loading textbook...")

reader = PdfReader("Introduction_to_Data_Science.pdf")

text = ""

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text

print("Total characters:", len(text))

incorrect startxref pointer(1)
parsing for Object Streams


Loading textbook...
Total characters: 3427


In [8]:
# ----------------------------------------------------------
# STEP 6 — Chunk the Text
# ----------------------------------------------------------

def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = chunk_text(text)

print("Total chunks created:", len(chunks))

Total chunks created: 8


In [9]:
# ----------------------------------------------------------
# STEP 7 — Create Embeddings and Store in Chroma
# ----------------------------------------------------------

print("Creating embeddings...")

for i, chunk in enumerate(chunks):

    embedding = embedding_model.encode(chunk).tolist()

    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("All chunks stored in vector database")

Creating embeddings...
All chunks stored in vector database


In [10]:
# ----------------------------------------------------------
# STEP 8 — Retriever
# ----------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]

In [11]:
# ----------------------------------------------------------
# STEP 9 — Answer Generation
# ----------------------------------------------------------

def answer_question(query):

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
Context:
{context}

Question: {query}

Answer the question clearly using only the context.
"""

    response = llm.chat.complete(
        model="mistral-small-latest",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    return response.choices[0].message.content


In [12]:
# ----------------------------------------------------------
# STEP 10 — Chat Loop
# ----------------------------------------------------------

print("\n==============================")
print("University Library Assistant")
print("Type 'exit' to stop")
print("==============================\n")

while True:

    question = input("Ask a question: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    answer = answer_question(question)

    print("\nAnswer:\n", answer)
    print("\n" + "-"*60 + "\n")


University Library Assistant
Type 'exit' to stop


Answer:
 The **Data Science Workflow** consists of the following steps:

1. **Define the problem or research question.**
2. **Collect relevant data** from databases, APIs, or files.
3. **Clean and preprocess the data.**
4. **Explore the data** using statistics and visualization.
5. **Build predictive or analytical models.**
6. **Evaluate and interpret results.**
7. **Communicate insights to stakeholders.**

This structured approach ensures systematic analysis and meaningful insights from data.

------------------------------------------------------------

Goodbye!
